<a href="https://colab.research.google.com/github/slomi23/ML_fx/blob/main/model_experiment_PatchTST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!git clone "https://github.com/slomi23/ML_fx.git"
!cd ML_fx/

fatal: destination path 'ML_fx' already exists and is not an empty directory.


In [12]:
import pandas as pd
import numpy as np
import os
import zipfile
import io

PROCCESSED_DATA_DIR = "./ML_fx/data/processed/"
train=pd.read_csv(os.path.join(PROCCESSED_DATA_DIR, "train_prepared.csv"))
print(train.head())

   Store  Dept        Date  Weekly_Sales  IsHoliday  Temperature  Fuel_Price  \
0      1     1  2010-02-05      24924.50          0        42.31       2.572   
1      1     1  2010-02-12      46039.49          1        38.51       2.548   
2      1     1  2010-02-19      41595.55          0        39.93       2.514   
3      1     1  2010-02-26      19403.54          0        46.63       2.561   
4      1     1  2010-03-05      21827.90          0        46.50       2.625   

   MarkDown1  MarkDown2  MarkDown3  ...  Type    Size  sales_lag_52  Year  \
0    5347.45      192.0       24.6  ...    20  151315           0.0  2010   
1    5347.45      192.0       24.6  ...    20  151315           0.0  2010   
2    5347.45      192.0       24.6  ...    20  151315           0.0  2010   
3    5347.45      192.0       24.6  ...    20  151315           0.0  2010   
4    5347.45      192.0       24.6  ...    20  151315           0.0  2010   

   month_sin     month_cos   dow_sin   dow_cos  week_sin

In [13]:
split_date = '2011-12-01'

# Create Train and Validation Sets
val_set = train[train['Date'] >= split_date]
train_set = train[train['Date'] < split_date]

y_train = train_set['Weekly_Sales']
X_train = train_set.drop(columns=['Weekly_Sales', 'Date'])
y_val = val_set['Weekly_Sales']
X_val = val_set.drop(columns=['Weekly_Sales', 'Date'])


print(f"Final Training Set Shape: {train_set.shape}")
print(f"Validation Set Shape: {val_set.shape}")
print(f"Validation Period: {val_set['Date'].min()} to {val_set['Date'].max()}")

Final Training Set Shape: (279085, 24)
Validation Set Shape: (142485, 24)
Validation Period: 2011-12-02 to 2012-10-26


In [14]:
!pip install wandb -q
!pip install neuralforecast pytorch-lightning wandb -q

import wandb
import os

# Retrieve the secret from Kaggle Secrets

api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


In [15]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import wandb
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MAE
from neuralforecast.models import PatchTST
from pytorch_lightning.callbacks import Callback
from sklearn.metrics import mean_absolute_error

run = wandb.init(
    project="ML_fx_PatchTST_Walmart",
    name="PatchTST_run2_different_h",
    config={
        "input_size": 52,
        "h": 39,
        "patch_len": 8,
        "stride": 4,
        "encoder_layers": 3,
        "n_heads": 4,
        "hidden_size": 128,
        "learning_rate": 1e-3,
        "max_steps": 1000,
        "val_check_steps": 50,
        "batch_size": 128,
        "random_seed": 42,
        "val_size_weeks": 39,
    },
)

df_all = pd.concat([train_set, val_set], ignore_index=True)
df_all["unique_id"] = df_all["Store"].astype(str) + "_" + df_all["Dept"].astype(str)
df_all["ds"] = pd.to_datetime(df_all["Date"])
df_all = df_all.rename(columns={"Weekly_Sales": "y"})

df_nf = (
    df_all[["unique_id", "ds", "y", "IsHoliday"]]
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

n_test_weeks = df_nf.loc[df_nf["ds"] >= pd.Timestamp(split_date), "ds"].nunique()
n_val_weeks = run.config["val_size_weeks"]

print(f"Holdout (test) window: {n_test_weeks} weeks starting {split_date}")
print(f"Internal validation window for early stopping: {n_val_weeks} weeks")

test_mask = df_nf["ds"] >= pd.Timestamp(split_date)
test_counts = df_nf[test_mask].groupby("unique_id").size()
pre_counts = df_nf[~test_mask].groupby("unique_id").size()

full_test_ids = set(test_counts[test_counts == n_test_weeks].index)
enough_history_ids = set(
    pre_counts[pre_counts >= run.config["input_size"] + n_val_weeks].index
)
keep_ids = full_test_ids & enough_history_ids

n_total_ids = df_nf["unique_id"].nunique()
print(
    f"Keeping {len(keep_ids)} of {n_total_ids} series "
    f"({n_total_ids - len(keep_ids)} dropped: intermittent Store/Dept combos "
    "lacking full test-window coverage or enough history before it)"
)

df_nf = df_nf[df_nf["unique_id"].isin(keep_ids)].reset_index(drop=True)

class WandbEpochLogger(Callback):

  def on_validation_epoch_end(self, trainer, pl_module):
    metrics = trainer.callback_metrics
    val_loss = metrics.get("val_loss")
    train_loss = metrics.get("train_loss")

    log_dict = {"step": trainer.global_step}
    if val_loss is not None:
      log_dict["epoch_val_mae"] = val_loss.item()
    if train_loss is not None:
      log_dict["epoch_train_mae"] = train_loss.item()

    if len(log_dict) > 1:
      wandb.log(log_dict)


patchtst_model = PatchTST(
    h=run.config["h"],
    input_size=run.config["input_size"],
    patch_len=run.config["patch_len"],
    stride=run.config["stride"],
    encoder_layers=run.config["encoder_layers"],
    n_heads=run.config["n_heads"],
    hidden_size=run.config["hidden_size"],
    loss=MAE(),
    learning_rate=run.config["learning_rate"],
    max_steps=run.config["max_steps"],
    val_check_steps=run.config["val_check_steps"],
    batch_size=run.config["batch_size"],
    random_seed=run.config["random_seed"],
    start_padding_enabled=True,
    callbacks=[WandbEpochLogger()],
)

nf = NeuralForecast(models=[patchtst_model], freq="W-FRI")
print("Training PatchTST model with walk-forward validation...")
cv_df = nf.cross_validation(
    df=df_nf,
    val_size=n_val_weeks,
    test_size=n_test_weeks,
    n_windows=None,
    step_size=1,
)

print("Evaluating predictions...")
val_eval = cv_df.merge(
    df_nf[["unique_id", "ds", "IsHoliday"]], on=["unique_id", "ds"], how="left"
).dropna(subset=["y", "PatchTST"])

y_true = val_eval["y"].values
y_pred = val_eval["PatchTST"].values
is_holiday = val_eval["IsHoliday"].values

weights = np.where(is_holiday == 1, 5, 1)
final_wmae = np.average(np.abs(y_true - y_pred), weights=weights)
final_mae = mean_absolute_error(y_true, y_pred)

run.summary["best_val_wmae"] = final_wmae
run.summary["best_val_mae"] = final_mae
wandb.log({"final_val_wmae": final_wmae, "final_val_mae": final_mae})

artifact = wandb.Artifact("patchtst-model", type="model")
os.makedirs("patchtst_checkpoint", exist_ok=True)
nf.save(path="./patchtst_checkpoint/", overwrite=True)
artifact.add_dir("./patchtst_checkpoint/")
run.log_artifact(artifact)

run.link_artifact(
    artifact,
    target_path="wandb-registry-model/Walmart-PatchTST",
    aliases=["latest"]
)

print("--------------------------------------------------")
print(f"Final Validation MAE:  {final_mae:.2f}")
print(f"Final Validation WMAE: {final_wmae:.2f}")
print("--------------------------------------------------")



wandb.finish()


Holdout (test) window: 48 weeks starting 2011-12-01
Internal validation window for early stopping: 39 weeks
Keeping 2692 of 3331 series (639 dropped: intermittent Store/Dept combos lacking full test-window coverage or enough history before it)


INFO:lightning_fabric.utilities.seed:Seed set to 42


Training PatchTST model with walk-forward validation...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 465 K  | train
------------------------------------------------------------------
465 K     Trainable params
3         Non-trainable params
465 K     Total params


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Predicting: |          | 0/? [00:00<?, ?it/s]

Evaluating predictions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
wandb: Adding directory to artifact (patchtst_checkpoint)... Done. 0.0s


--------------------------------------------------
Final Validation MAE:  3507.63
Final Validation WMAE: 3472.83
--------------------------------------------------


epoch_train_mae,▂▃▂▂▂▂▂▂▂▁█▂▁▂▂▂▁▂▂▁
final_val_mae,▁
final_val_wmae,▁
step,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
best_val_mae,3507.63458
best_val_wmae,3472.83201
epoch_train_mae,2960.86182
final_val_mae,3507.63458
final_val_wmae,3472.83201
step,1000
